# Aurora fio `rand_write` — Engine Statistics

Per-engine run statistics (N, mean, min, median, max) for:\n**Bandwidth** (GiB/s), **IOPS**, **Mean latency** (ms), **CPU** (usr+sys %).\n\nCommon configuration: `bs=1m`, `numjobs=16`, `iodepth=16`, `rw=randwrite`, `runtime=60s`.

## DFS

In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path(".")

In [2]:
def parse_dfs_file(fpath):
    m = re.search(r"fio_dfs_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json", Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))

    with open(fpath) as f:
        d = json.load(f)

    job = d["jobs"][0]          # group_reporting=1 → single aggregated job
    w   = job["write"]

    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
    )


rows = [parse_dfs_file(p) for p in sorted(RESULTS_DIR.glob("fio_dfs_*.json"))]
df   = pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)
df.index.name = "run"
df.index += 1

print(f"Loaded {len(df)} DFS runs")
df

Loaded 10 DFS runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781795356,1m,16,16,88.244622,90362.492500,2.706695,88.248364
2,1781795431,1m,16,16,84.629335,86660.439304,2.819271,93.699441
3,1781795506,1m,16,16,85.147631,87191.173775,2.823136,76.092669
4,1781795581,1m,16,16,88.007107,90119.277369,2.701697,96.776593
5,1781795656,1m,16,16,88.119113,90233.971635,2.701460,90.277601
6,1781795731,1m,16,16,79.368342,81273.181788,2.992762,99.321050
7,1781795805,1m,16,16,88.987510,91123.210506,2.678237,90.089925
8,1781795880,1m,16,16,88.426185,90548.413439,2.688792,96.152764
9,1781795955,1m,16,16,87.735392,89841.041281,2.704861,98.251791


### Summary Statistics

In [3]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"    : "CPU (%)",
}

stats = (
    df[list(METRICS)]
    .agg(["count", "mean", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats["N"] = stats["N"].astype(int)

pd.set_option("display.float_format", "{:.4f}".format)
stats

,N,mean,min,median,max
BW (GiB/s),10,86.7165,79.3683,88.0631,88.9875
IOPS,10,88797.6537,81273.1818,90176.6245,91123.2105
Mean Latency (ms),10,2.7505,2.6782,2.7033,2.9928
CPU (%),10,92.2300,76.0927,93.5446,99.3210


## Interception libraries

### libioil / libaio

Files that begin with fio error/signal lines are parsed by skipping to the first `{`;\nabnormally terminated runs are flagged in the `note` column.

In [4]:
def parse_libioil_libaio_file(fpath):
    m = re.search(r"fio_libioil_libaio_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json", Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))

    text = Path(fpath).read_text()
    json_start = text.find("{")
    d    = json.loads(text[json_start:])

    job  = d["jobs"][0]
    w    = job["write"]
    note = "aborted" if text[:json_start].strip() else ""

    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
        note        = note,
    )


rows_il, skipped = [], []
for p in sorted(RESULTS_DIR.glob("fio_libioil_libaio_*.json")):
    try:
        rows_il.append(parse_libioil_libaio_file(p))
    except Exception as e:
        skipped.append((p.name, str(e)))

df_il = pd.DataFrame(rows_il).sort_values("timestamp").reset_index(drop=True)
df_il.index.name = "run"
df_il.index += 1

if skipped:
    print(f"Skipped {len(skipped)} file(s): {[n for n,_ in skipped]}")
print(f"Loaded {len(df_il)} libioil/libaio runs")
df_il

Loaded 10 libioil/libaio runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu,note
run,,,,,,,,,
1,1781728493,1m,16,16,14.3121,14655.5506,17.1489,23.7958,
2,1781796475,1m,16,16,10.9574,11220.3427,22.4472,20.3733,
3,1781796553,1m,16,16,11.1688,11436.8188,22.0908,15.3347,
4,1781796631,1m,16,16,11.2066,11475.5175,22.0052,16.1561,
5,1781796710,1m,16,16,10.6864,10942.8686,23.0332,19.2534,
6,1781796789,1m,16,16,10.7203,10977.5507,22.9517,19.7184,
7,1781796867,1m,16,16,11.1938,11462.4846,22.0347,15.7706,
8,1781796951,1m,16,16,6.3494,6501.7666,35.2316,10.6497,
9,1781797061,1m,16,16,10.8537,11114.2295,22.6760,19.3636,


In [5]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"        : "CPU (%)",
}

stats_il = (
    df_il[list(METRICS)]
    .agg(["count", "mean", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_il["N"] = stats_il["N"].astype(int)

pd.set_option("display.float_format", "{:.4f}".format)
stats_il

,N,mean,min,median,max
BW (GiB/s),10,10.8657,6.3494,11.0631,14.3121
IOPS,10,11126.5064,6501.7666,11328.5807,14655.5506
Mean Latency (ms),10,23.1624,17.1489,22.2690,35.2316
CPU (%),10,17.6238,10.6497,17.7047,23.7958


---
## libioil / pvsync2

In [6]:
def parse_simple(fpath, pattern):
    m = re.search(pattern, Path(fpath).name)
    bs, nj, iod, ts = m.group(1), int(m.group(2)), int(m.group(3)), int(m.group(4))
    text = Path(fpath).read_text()
    d    = json.loads(text[text.find("{"):])
    job  = d["jobs"][0]
    w    = job["write"]
    return dict(
        timestamp   = ts,
        block_size  = bs,
        numjobs     = nj,
        iodepth     = iod,
        bw_GiBs     = w["bw_bytes"] / 1024**3,
        iops        = w["iops"],
        lat_mean_ms = w["lat_ns"]["mean"] / 1e6,
        cpu         = job["usr_cpu"] + job["sys_cpu"],
    )

PATTERN = r"fio_\S+?_bs(\S+?)_nj(\d+)_iod(\d+)_(\d+)\.json"

rows_pv2 = [parse_simple(p, PATTERN)
            for p in sorted(RESULTS_DIR.glob("fio_libioil_pvsync2_*.json"))]
df_pv2 = pd.DataFrame(rows_pv2).sort_values("timestamp").reset_index(drop=True)
df_pv2.index.name = "run"
df_pv2.index += 1

print(f"Loaded {len(df_pv2)} libioil/pvsync2 runs")
df_pv2

Loaded 10 libioil/pvsync2 runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781799098,1m,16,16,12.1915,12484.1339,1.1111,13.0670
2,1781799185,1m,16,16,12.1465,12437.9687,1.1134,13.6493
3,1781799271,1m,16,16,12.3258,12621.5959,1.1274,11.9604
4,1781799350,1m,16,16,12.0796,12369.4877,1.1347,13.3646
5,1781799429,1m,16,16,12.1441,12435.5688,1.1283,13.6521
6,1781799507,1m,16,16,12.2195,12512.7291,1.1265,13.0730
7,1781799592,1m,16,16,12.3051,12600.4300,1.1223,12.5901
8,1781799678,1m,16,16,12.2717,12566.1811,1.1237,12.6638
9,1781799768,1m,16,16,12.2494,12543.4319,1.1306,12.1827


In [7]:
METRICS = {
    "bw_GiBs"    : "BW (GiB/s)",
    "iops"       : "IOPS",
    "lat_mean_ms": "Mean Latency (ms)",
    "cpu"        : "CPU (%)",
}

stats_pv2 = (
    df_pv2[list(METRICS)]
    .agg(["count", "mean", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_pv2["N"] = stats_pv2["N"].astype(int)

pd.set_option("display.float_format", "{:.4f}".format)
stats_pv2

,N,mean,min,median,max
BW (GiB/s),10,12.2120,12.0796,12.2055,12.3258
IOPS,10,12505.0494,12369.4877,12498.4315,12621.5959
Mean Latency (ms),10,1.1258,1.1111,1.1269,1.1398
CPU (%),10,12.8424,11.9604,12.8654,13.6521


---
### libioil / psync

In [8]:
rows_ps = [parse_simple(p, PATTERN)
           for p in sorted(RESULTS_DIR.glob("fio_libioil_psync_*.json"))]
df_ps = pd.DataFrame(rows_ps).sort_values("timestamp").reset_index(drop=True)
df_ps.index.name = "run"
df_ps.index += 1

print(f"Loaded {len(df_ps)} libioil/psync runs")
df_ps

Loaded 10 libioil/psync runs


,timestamp,block_size,numjobs,iodepth,bw_GiBs,iops,lat_mean_ms,cpu
run,,,,,,,,
1,1781798270,1m,16,16,18.6840,19132.4145,0.7754,99.3272
2,1781798355,1m,16,16,13.0376,13350.4973,0.9036,80.1125
3,1781798445,1m,16,16,20.2395,20725.2712,0.7142,99.5727
4,1781798524,1m,16,16,19.6942,20166.8972,0.7349,99.5978
5,1781798603,1m,16,16,20.3258,20813.5864,0.7101,99.5868
6,1781798682,1m,16,16,19.4974,19965.3839,0.7434,99.5946
7,1781798760,1m,16,16,21.0228,21527.3579,0.6850,99.5866
8,1781798839,1m,16,16,20.8746,21375.5937,0.6896,99.5773
9,1781798924,1m,16,16,19.4016,19867.2689,0.7344,98.1060


In [9]:
stats_ps = (
    df_ps[list(METRICS)]
    .agg(["count", "mean", "min", "median", "max"])
    .rename(index={"count": "N"})
    .rename(columns=METRICS)
    .T
)
stats_ps["N"] = stats_ps["N"].astype(int)

pd.set_option("display.float_format", "{:.4f}".format)
stats_ps

,N,mean,min,median,max
BW (GiB/s),10,19.2252,13.0376,19.5958,21.0228
IOPS,10,19686.5639,13350.4973,20066.1406,21527.3579
Mean Latency (ms),10,0.7424,0.6850,0.7341,0.9036
CPU (%),10,97.3406,80.1125,99.5750,99.5978


---
### Stability Summary — libioil Engines

| Engine | N | BW mean (GiB/s) | BW median | BW min | BW max | Lat mean (ms) | Assessment |
|---|---|---|---|---|---|---|---|
| libioil/libaio  | 10 | 10.87 | 11.06 | 6.35 | 14.31 | 23.2 | **Unstable** — two outliers (run 1 from prior day at 14.3, run 8 aborted at 6.4); core 8 runs cluster at 10.7–11.2 GiB/s |
| libioil/pvsync2 | 10 | 12.21 | 12.21 | 12.08 | 12.33 |  1.1 | **Very stable** — <1% CV across all runs, negligible variance |
| libioil/psync   | 10 | 19.23 | 19.60 | 13.04 | 21.02 |  0.7 | **Mostly stable** — one outlier run at 13.0 GiB/s (~34% below); remaining 9 runs tight at 19.5–21.0 GiB/s |

**libioil/psync delivers the highest bandwidth** (~19.5 GiB/s typical) at the lowest latency (~0.73 ms),
at the cost of near-100% CPU utilization (synchronous, CPU-bound I/O path).
**libioil/pvsync2** is the most repeatable but achieves only ~12.2 GiB/s.
**libioil/libaio** sits in between in bandwidth but shows the most run-to-run variation,
partly due to an aborted run and an earlier isolated test captured in the result set.